In [ ]:
# ===================== Cell 1: input representation + model =====================
# Drug: RDKit 3D conformer (drug_sdf/sdf/<did>.sdf); heavy atoms are nodes, fully connected edges (edge feature = bond type)
# Protein: AF3 predicted structure (af3_protein/pred/<pid>/<pid>_model.cif); Cα atoms are nodes, kNN edges (edge feature = sequence separation)
# Each side is encoded by its own EGNN (E(3)-equivariant GNN), keeping only the invariant node features h,
# which are fused by DrugBAN's bilinear attention network (BAN, DrugBAN's BANLayer unchanged); DrugBAN's MLP decoder outputs the binding logit.
import hashlib, json, math, os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem, RDLogger

RDLogger.DisableLog('rdApp.*')
DTI = '/work/xujian/data/linxq/dti'
md5 = lambda s: hashlib.md5(s.encode()).hexdigest()[:16]   # drug/protein id, same as in the data preparation scripts

# ---------- Drug: SDF -> graph ----------
ELEM = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P', 'B', 'Si', 'Se']
HYB = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2, Chem.HybridizationType.SP3,
       Chem.HybridizationType.SP3D, Chem.HybridizationType.SP3D2]
BOND = {Chem.BondType.SINGLE: 1, Chem.BondType.DOUBLE: 2, Chem.BondType.TRIPLE: 3, Chem.BondType.AROMATIC: 4}
ATOM_VOCAB = [len(ELEM) + 1, 7, 5, 2, len(HYB) + 1, 5, 2]   # element, degree, charge, aromatic, hybridization, num H, in ring
N_BOND = 5                                                  # no bond / single / double / triple / aromatic


def drug_graph(sdf_path):
    mol = Chem.MolFromMolFile(sdf_path, removeHs=True)
    pos = torch.tensor(mol.GetConformer().GetPositions(), dtype=torch.float32)
    feat = [[ELEM.index(a.GetSymbol()) if a.GetSymbol() in ELEM else len(ELEM),
             min(a.GetDegree(), 6), min(max(a.GetFormalCharge(), -2), 2) + 2, int(a.GetIsAromatic()),
             HYB.index(a.GetHybridization()) if a.GetHybridization() in HYB else len(HYB),
             min(a.GetTotalNumHs(), 4), int(a.IsInRing())] for a in mol.GetAtoms()]
    n = len(feat)
    bond = torch.zeros(n, n, dtype=torch.long)
    for b in mol.GetBonds():
        i, j, t = b.GetBeginAtomIdx(), b.GetEndAtomIdx(), BOND.get(b.GetBondType(), 1)
        bond[i, j] = bond[j, i] = t
    src, dst = (~torch.eye(n, dtype=torch.bool)).nonzero(as_tuple=True)   # fully connected (no self loops)
    return dict(pos=pos - pos.mean(0), feat=torch.tensor(feat, dtype=torch.long),
                edge=torch.stack([src, dst]), eattr=bond[src, dst])


# ---------- Protein: AF3 mmCIF -> graph ----------
AA = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
      'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']
K_NN = 16
SEP_BINS = [1, 2, 3, 4, 8, 16, 32]     # sequence separation |i-j| binned -> 8 classes


def protein_graph(cif_path):
    cols, ca = [], []
    for line in open(cif_path):
        if line.startswith('_atom_site.'):
            cols.append(line.strip().split('.', 1)[1])
        elif line.startswith('ATOM') and cols:
            f = dict(zip(cols, line.split()))
            if f['label_atom_id'] == 'CA':
                ca.append((AA.index(f['label_comp_id']) if f['label_comp_id'] in AA else 20,
                           float(f['Cartn_x']), float(f['Cartn_y']), float(f['Cartn_z']),
                           float(f['B_iso_or_equiv'])))
    a = torch.tensor(ca, dtype=torch.float32)
    pos = a[:, 1:4] - a[:, 1:4].mean(0)
    n, k = len(pos), min(K_NN, len(pos) - 1)
    nbr = torch.cdist(pos, pos).topk(k + 1, largest=False).indices[:, 1:]     # k nearest neighbours of each residue
    dst = torch.arange(n).repeat_interleave(k)
    src = nbr.reshape(-1)
    sep = torch.bucketize((src - dst).abs(), torch.tensor(SEP_BINS))
    return dict(pos=pos, feat=a[:, 0].long(), plddt=a[:, 4] / 100, edge=torch.stack([src, dst]), eattr=sep)


# ---------- EGNN ----------
class RBF(nn.Module):
    def __init__(self, n=32, cutoff=20.0):
        super().__init__()
        self.register_buffer('mu', torch.linspace(0, cutoff, n))
        self.gamma = (n / cutoff) ** 2

    def forward(self, d):
        return torch.exp(-self.gamma * (d - self.mu) ** 2)


class EGNNLayer(nn.Module):
    """Satorras et al. 2021: messages depend only on distances (invariant); coordinates are updated along relative vectors (equivariant)."""

    def __init__(self, dim, edim, n_rbf=32):
        super().__init__()
        self.rbf = RBF(n_rbf)
        self.msg = nn.Sequential(nn.Linear(2 * dim + n_rbf + edim, dim), nn.SiLU(), nn.Linear(dim, dim), nn.SiLU())
        self.gate = nn.Sequential(nn.Linear(dim, 1), nn.Sigmoid())
        self.coord = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, 1, bias=False))
        nn.init.xavier_uniform_(self.coord[-1].weight, gain=1e-3)
        self.upd = nn.Sequential(nn.Linear(2 * dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.norm = nn.LayerNorm(dim)

    def forward(self, h, x, edge, e):
        src, dst = edge
        rel = x[dst] - x[src]
        dist = rel.norm(dim=-1, keepdim=True)
        m = self.msg(torch.cat([h[dst], h[src], self.rbf(dist), e], -1))
        m = m * self.gate(m)
        deg = torch.zeros(len(h), 1, device=h.device).index_add_(0, dst, torch.ones_like(dist)).clamp(min=1)
        x = x + torch.zeros_like(x).index_add_(0, dst, rel / (dist + 1) * torch.tanh(self.coord(m))) / deg
        h = self.norm(h + self.upd(torch.cat([h, torch.zeros_like(h).index_add_(0, dst, m) / deg], -1)))
        return h, x


class EGNN(nn.Module):
    def __init__(self, dim, edim, n_layers):
        super().__init__()
        self.layers = nn.ModuleList([EGNNLayer(dim, edim) for _ in range(n_layers)])

    def forward(self, h, x, edge, e):
        for layer in self.layers:
            h, x = layer(h, x, edge, e)
        return h


class DrugEncoder(nn.Module):
    def __init__(self, dim, n_layers):
        super().__init__()
        self.emb = nn.ModuleList([nn.Embedding(v, dim) for v in ATOM_VOCAB])
        self.bond = nn.Embedding(N_BOND, 16)
        self.egnn = EGNN(dim, 16, n_layers)

    def forward(self, g):
        h = sum(emb(g['feat'][:, i]) for i, emb in enumerate(self.emb))
        return self.egnn(h, g['pos'], g['edge'], self.bond(g['eattr']))


class ProteinEncoder(nn.Module):
    def __init__(self, dim, n_layers):
        super().__init__()
        self.aa = nn.Embedding(21, dim)
        self.plddt = nn.Linear(1, dim)
        self.sep = nn.Embedding(len(SEP_BINS) + 1, 16)
        self.egnn = EGNN(dim, 16, n_layers)

    def forward(self, g):
        h = self.aa(g['feat']) + self.plddt(g['plddt'][:, None])
        return self.egnn(h, g['pos'], g['edge'], self.sep(g['eattr']))


def to_dense(h, batch, idx, B):
    """Concatenated node features -> zero-padded (B, Nmax, D) + mask, for the bilinear attention."""
    out = h.new_zeros(B, int(idx.max()) + 1, h.size(-1))
    mask = torch.zeros(B, out.size(1), dtype=torch.bool, device=h.device)
    out[batch, idx] = h
    mask[batch, idx] = True
    return out, mask


# ---------- Bilinear attention network + decoder: DrugBAN's code, unchanged ----------
# From DrugBAN (https://github.com/peizhenbai/DrugBAN, ban.py and models.py), MIT License, Copyright (c) 2022 peizhenbai.
# BANLayer / FCNet are themselves modified from ban-vqa (https://github.com/jnhwkim/ban-vqa).
from torch.nn.utils.weight_norm import weight_norm


class BANLayer(nn.Module):
    def __init__(self, v_dim, q_dim, h_dim, h_out, act='ReLU', dropout=0.2, k=3):
        super(BANLayer, self).__init__()

        self.c = 32
        self.k = k
        self.v_dim = v_dim
        self.q_dim = q_dim
        self.h_dim = h_dim
        self.h_out = h_out

        self.v_net = FCNet([v_dim, h_dim * self.k], act=act, dropout=dropout)
        self.q_net = FCNet([q_dim, h_dim * self.k], act=act, dropout=dropout)
        # self.dropout = nn.Dropout(dropout[1])
        if 1 < k:
            self.p_net = nn.AvgPool1d(self.k, stride=self.k)

        if h_out <= self.c:
            self.h_mat = nn.Parameter(torch.Tensor(1, h_out, 1, h_dim * self.k).normal_())
            self.h_bias = nn.Parameter(torch.Tensor(1, h_out, 1, 1).normal_())
        else:
            self.h_net = weight_norm(nn.Linear(h_dim * self.k, h_out), dim=None)

        self.bn = nn.BatchNorm1d(h_dim)

    def attention_pooling(self, v, q, att_map):
        fusion_logits = torch.einsum('bvk,bvq,bqk->bk', (v, att_map, q))
        if 1 < self.k:
            fusion_logits = fusion_logits.unsqueeze(1)  # b x 1 x d
            fusion_logits = self.p_net(fusion_logits).squeeze(1) * self.k  # sum-pooling
        return fusion_logits

    def forward(self, v, q, softmax=False):
        v_num = v.size(1)
        q_num = q.size(1)
        if self.h_out <= self.c:
            v_ = self.v_net(v)
            q_ = self.q_net(q)
            att_maps = torch.einsum('xhyk,bvk,bqk->bhvq', (self.h_mat, v_, q_)) + self.h_bias
        else:
            v_ = self.v_net(v).transpose(1, 2).unsqueeze(3)
            q_ = self.q_net(q).transpose(1, 2).unsqueeze(2)
            d_ = torch.matmul(v_, q_)  # b x h_dim x v x q
            att_maps = self.h_net(d_.transpose(1, 2).transpose(2, 3))  # b x v x q x h_out
            att_maps = att_maps.transpose(2, 3).transpose(1, 2)  # b x h_out x v x q
        if softmax:
            p = nn.functional.softmax(att_maps.view(-1, self.h_out, v_num * q_num), 2)
            att_maps = p.view(-1, self.h_out, v_num, q_num)
        logits = self.attention_pooling(v_, q_, att_maps[:, 0, :, :])
        for i in range(1, self.h_out):
            logits_i = self.attention_pooling(v_, q_, att_maps[:, i, :, :])
            logits += logits_i
        logits = self.bn(logits)
        return logits, att_maps


class FCNet(nn.Module):
    """Simple class for non-linear fully connect network
    Modified from https://github.com/jnhwkim/ban-vqa/blob/master/fc.py
    """

    def __init__(self, dims, act='ReLU', dropout=0):
        super(FCNet, self).__init__()

        layers = []
        for i in range(len(dims) - 2):
            in_dim = dims[i]
            out_dim = dims[i + 1]
            if 0 < dropout:
                layers.append(nn.Dropout(dropout))
            layers.append(weight_norm(nn.Linear(in_dim, out_dim), dim=None))
            if '' != act:
                layers.append(getattr(nn, act)())
        if 0 < dropout:
            layers.append(nn.Dropout(dropout))
        layers.append(weight_norm(nn.Linear(dims[-2], dims[-1]), dim=None))
        if '' != act:
            layers.append(getattr(nn, act)())

        self.main = nn.Sequential(*layers)

    def forward(self, x):
        return self.main(x)


class MLPDecoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, binary=1):
        super(MLPDecoder, self).__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, out_dim)
        self.bn3 = nn.BatchNorm1d(out_dim)
        self.fc4 = nn.Linear(out_dim, binary)

    def forward(self, x):
        x = self.bn1(F.relu(self.fc1(x)))
        x = self.bn2(F.relu(self.fc2(x)))
        x = self.bn3(F.relu(self.fc3(x)))
        x = self.fc4(x)
        return x


class E3DTI(nn.Module):
    """EGNN drug / protein encoders + DrugBAN's bilinear attention and MLP decoder.
    BAN hyperparameters follow DrugBAN's config: h_dim = DECODER.IN_DIM = 256, h_out = BCN.HEADS = 2, k = 3,
    dropout 0.2, no softmax, weight norm on h_mat; decoder 256 -> 512 -> 512 -> 128 -> 1."""

    def __init__(self, dim=128, drug_layers=4, prot_layers=4, ban_dim=256, heads=2, mlp_hidden=512, mlp_out=128):
        super().__init__()
        self.drug = DrugEncoder(dim, drug_layers)
        self.prot = ProteinEncoder(dim, prot_layers)
        self.bcn = weight_norm(BANLayer(v_dim=dim, q_dim=dim, h_dim=ban_dim, h_out=heads), name='h_mat', dim=None)
        self.mlp_classifier = MLPDecoder(ban_dim, mlp_hidden, mlp_out, binary=1)

    def forward(self, gd, gp):
        B = int(gd['n_graphs'])
        v_d, _ = to_dense(self.drug(gd), gd['batch'], gd['idx'], B)    # zero-padded, as DrugBAN pads drugs/proteins
        v_p, _ = to_dense(self.prot(gp), gp['batch'], gp['idx'], B)
        f, att = self.bcn(v_d, v_p)
        return self.mlp_classifier(f).squeeze(-1), att


def collate_graphs(gs):
    """Merge a batch of graphs into one: concatenate nodes, offset edge indices, and record each node's graph (batch) and position within its graph (idx)."""
    n = torch.tensor([len(g['pos']) for g in gs])
    off = torch.cumsum(n, 0) - n
    out = {k: torch.cat([g[k] for g in gs]) for k in gs[0] if k != 'edge'}
    out['edge'] = torch.cat([g['edge'] + o for g, o in zip(gs, off)], 1)
    out['batch'] = torch.arange(len(gs)).repeat_interleave(n)
    out['idx'] = torch.cat([torch.arange(k) for k in n.tolist()])
    out['n_graphs'] = torch.tensor(len(gs))
    return out


# Equivariance self-check: randomly rotate + translate the drug and the protein separately; the prediction should not change
_m = E3DTI().eval()
_gd = {'pos': torch.randn(9, 3) * 3, 'feat': torch.zeros(9, 7, dtype=torch.long)}
_gd['edge'] = (~torch.eye(9, dtype=torch.bool)).nonzero().T
_gd['eattr'] = torch.zeros(_gd['edge'].size(1), dtype=torch.long)
_gp = {'pos': torch.randn(30, 3) * 8, 'feat': torch.randint(0, 21, (30,)), 'plddt': torch.rand(30)}
_nb = torch.cdist(_gp['pos'], _gp['pos']).topk(K_NN + 1, largest=False).indices[:, 1:]
_gp['edge'] = torch.stack([_nb.reshape(-1), torch.arange(30).repeat_interleave(K_NN)])
_gp['eattr'] = torch.zeros(_gp['edge'].size(1), dtype=torch.long)
_rot = lambda g: {**g, 'pos': g['pos'] @ torch.linalg.qr(torch.randn(3, 3))[0] + torch.randn(3) * 10}
with torch.no_grad():
    _a = _m(collate_graphs([_gd, _gd]), collate_graphs([_gp, _gp]))[0]
    _b = _m(collate_graphs([_rot(_gd), _rot(_gd)]), collate_graphs([_rot(_gp), _rot(_gp)]))[0]
print(f'E(3) invariance check: max logit difference under rotation/translation/reflection {(_a - _b).abs().max().item():.2e}')
print(f'Model parameters: {sum(p.numel() for p in E3DTI().parameters()) / 1e6:.2f} M')


In [ ]:
# ===================== Cell 2: data + training + validation + test =====================
import copy
import pandas as pd
from multiprocessing import Pool
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix

CFG = dict(dataset='biosnap', split='random', epochs=50, batch=32, lr=2e-4, wd=1e-4, warmup_epochs=1,
           patience=10, dim=128, drug_layers=4, prot_layers=4, seed=42, workers=6,
           max_train_pairs=None)    # set to a small number for a smoke test
OUT = f"{DTI}/runs/e3dti_{CFG['dataset']}_{CFG['split']}"
os.makedirs(OUT, exist_ok=True)
torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True

# ---------- 1. Read the splits and match them to SDF / AF3 structures ----------
split_dir = f"{DTI}/DrugBAN/datasets/{CFG['dataset']}/{CFG['split']}"
dfs = {s: pd.read_csv(f'{split_dir}/{s}.csv') for s in ['train', 'val', 'test']}
sdf_path = lambda did: f'{DTI}/drug_sdf/sdf/{did}.sdf'
cif_path = lambda pid: f'{DTI}/af3_protein/pred/{pid}/{pid}_model.cif'
for s, df in dfs.items():
    df['did'], df['pid'] = df.SMILES.map(md5), df.Protein.map(md5)
    ok = df.did.map(lambda d: os.path.exists(sdf_path(d))) & df.pid.map(lambda p: os.path.exists(cif_path(p)))
    print(f'{s:5s}: {len(df)} pairs, {ok.sum()} with SDF+AF3 structures ({ok.mean():.1%}), positive rate {df.Y[ok].mean():.3f}')
    dfs[s] = df[ok].reset_index(drop=True)
if CFG['max_train_pairs']:
    dfs['train'] = dfs['train'].sample(min(CFG['max_train_pairs'], len(dfs['train'])), random_state=0)

# ---------- 2. Build graphs for the molecules/proteins used (cached on disk; later runs read the cache) ----------
def build_cache(ids, fn, path_fn, cache):
    graphs = torch.load(cache) if os.path.exists(cache) else {}
    todo = [i for i in ids if i not in graphs]
    if todo:
        with Pool(32) as pool:
            graphs.update(zip(todo, pool.map(fn, [path_fn(i) for i in todo], chunksize=16)))
        torch.save(graphs, cache)
    return graphs

t0 = time.time()
all_df = pd.concat(dfs.values())
os.makedirs(f'{DTI}/cache', exist_ok=True)
DRUG = build_cache(sorted(set(all_df.did)), drug_graph, sdf_path, f'{DTI}/cache/drug_graphs.pt')
PROT = build_cache(sorted(set(all_df.pid)), protein_graph, cif_path, f'{DTI}/cache/protein_graphs_k{K_NN}.pt')
print(f'Graph building {time.time() - t0:.0f}s: {all_df.did.nunique()} drugs (mean {np.mean([len(DRUG[d]["pos"]) for d in set(all_df.did)]):.0f} heavy atoms), '
      f'{all_df.pid.nunique()} proteins (mean {np.mean([len(PROT[p]["pos"]) for p in set(all_df.pid)]):.0f} residues)')


class PairSet(torch.utils.data.Dataset):
    def __init__(self, df):
        self.d, self.p, self.y = df.did.tolist(), df.pid.tolist(), df.Y.astype(np.float32).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return DRUG[self.d[i]], PROT[self.p[i]], self.y[i]


def collate(batch):
    gd, gp, y = zip(*batch)
    return collate_graphs(gd), collate_graphs(gp), torch.tensor(y)


loader = lambda s, shuffle: torch.utils.data.DataLoader(
    PairSet(dfs[s]), batch_size=CFG['batch'], shuffle=shuffle, collate_fn=collate,
    num_workers=CFG['workers'], persistent_workers=True, drop_last=shuffle)
dl = {'train': loader('train', True), 'val': loader('val', False), 'test': loader('test', False)}

# ---------- 3. Training ----------
model = E3DTI(CFG['dim'], CFG['drug_layers'], CFG['prot_layers']).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
total, warm = CFG['epochs'] * len(dl['train']), CFG['warmup_epochs'] * len(dl['train'])
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda t: min(1, (t + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(t, total) / total)))
to_dev = lambda g: {k: v.to(dev, non_blocking=True) for k, v in g.items()}


@torch.no_grad()
def predict(net, loader):
    net.eval()
    ys, ps = [], []
    for gd, gp, y in loader:
        logit, _ = net(to_dev(gd), to_dev(gp))
        ys.append(y.numpy()); ps.append(torch.sigmoid(logit).float().cpu().numpy())
    y, p = np.concatenate(ys), np.concatenate(ps)
    loss = F.binary_cross_entropy(torch.tensor(p).clamp(1e-7, 1 - 1e-7), torch.tensor(y)).item()
    return y, p, dict(auroc=roc_auc_score(y, p), auprc=average_precision_score(y, p), loss=loss)


history, best, best_state, bad = [], -1, None, 0
for ep in range(1, CFG['epochs'] + 1):
    model.train()
    t0, run_loss = time.time(), 0.0
    for gd, gp, y in dl['train']:
        logit, _ = model(to_dev(gd), to_dev(gp))
        loss = F.binary_cross_entropy_with_logits(logit, y.to(dev))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        run_loss += loss.item()
    _, _, val = predict(model, dl['val'])
    history.append(dict(epoch=ep, train_loss=run_loss / len(dl['train']), **{f'val_{k}': v for k, v in val.items()}))
    flag = ''
    if val['auroc'] > best:
        best, best_state, bad, flag = val['auroc'], copy.deepcopy(model.state_dict()), 0, ' *'
        torch.save({'model': best_state, 'cfg': CFG, 'epoch': ep}, f'{OUT}/best.pt')
    else:
        bad += 1
    print(f"ep {ep:3d} | train loss {history[-1]['train_loss']:.4f} | val loss {val['loss']:.4f} "
          f"AUROC {val['auroc']:.4f} AUPRC {val['auprc']:.4f} | {time.time() - t0:.0f}s{flag}", flush=True)
    if bad >= CFG['patience']:
        print(f'Validation AUROC has not improved for {bad} epochs; stopping early')
        break
pd.DataFrame(history).to_csv(f'{OUT}/history.csv', index=False)

# ---------- 4. Test (using the epoch with the best validation score) ----------
model.load_state_dict(best_state)
yv, pv, val = predict(model, dl['val'])
yt, pt, test = predict(model, dl['test'])
fpr, tpr, thr = roc_curve(yv, pv)                     # threshold chosen on the validation set (max F1), without looking at test labels
prec = tpr / (tpr + fpr + 1e-12)
f1v = 2 * prec * tpr / (prec + tpr + 1e-12)
th = float(thr[np.argmax(f1v)])
tn, fp, fn, tp = confusion_matrix(yt, pt >= th).ravel()
test.update(threshold=th, accuracy=(tp + tn) / len(yt), sensitivity=tp / (tp + fn), specificity=tn / (tn + fp),
            precision=tp / max(tp + fp, 1), f1=2 * tp / (2 * tp + fp + fn))
res = dict(cfg=CFG, best_epoch=int(np.argmax([h['val_auroc'] for h in history]) + 1),
           n_pairs={s: len(d) for s, d in dfs.items()}, val=val, test=test)
json.dump(res, open(f'{OUT}/results.json', 'w'), indent=1, default=float)
print(f"\nBest epoch {res['best_epoch']} | validation AUROC {val['auroc']:.4f} AUPRC {val['auprc']:.4f}")
print('Test: ' + ', '.join(f'{k} {v:.4f}' for k, v in test.items()))
print(f'Results saved to {OUT}')
